# Scraping Tambahan: Views -- dari Grid Profil (BUKAN buka tiap video satu-satu)

Pendekatan **BARU**, hasil temuan kamu: views (`data-e2e="video-views"`) ternyata udah
kelihatan langsung di **grid video di halaman profil** (yang muncul pas hover/scroll card
video), jadi **ga perlu buka tiap video satu-satu**. Ini jauh lebih cepat & lebih aman dari
captcha dibanding rencana sebelumnya.

Cakupan tetap sama: cuma video yang `video_id`-nya muncul di `comments.csv`.

**Cara kerja:**
1. Buka halaman profil `@username` (bukan halaman video).
2. Scroll grid-nya pelan-pelan.
3. Tiap card video punya link `<a href=".../video/{video_id}">` yang di dalemnya ada
   `[data-e2e="video-views"]` -- jadi tiap scroll, kita kumpulin pasangan (video_id, views)
   dari SEMUA card yang kelihatan, lalu disaring cuma yang video_id-nya ada di target kita.
4. Berhenti scroll kalau semua video target punya username itu udah ketemu, ATAU udah mentok
   bawah (ga ada card baru beberapa kali scroll berturut-turut).

**PENTING -- tetep ada Cell E-TEST**: nyobain ke 1 profil dulu, nge-print mentah pasangan
(video_id, views) yang ketemu -- BELUM disimpen ke CSV -- biar kamu bisa cross-check manual ke
layar sebelum lanjut ke scraping massal semua username.

| Blok | Isi |
|---|---|
| **A** | Config path, field, delay |
| **B** | Telegram alert + audio warning |
| **C** | Tentuin target: grup `video_id` (dari `comments.csv`) per `username` |
| **D** | Resume helper (`video_views.csv`) + captcha check |
| **E** | Fungsi extract dari grid + scroll loop + **Cell E-TEST** |
| **F** | Loop scraping semua username (MAIN) |

---
## BLOK A — Config

In [1]:
# ==== Cell A1: Import & Config Path ====
import asyncio
import csv
import os
import sys
import random
import re
import json as jsonlib
import pygame
import requests
from datetime import datetime

import pandas as pd
from playwright.async_api import async_playwright


class CaptchaTimeoutError(Exception):
    """Dilempar saat captcha/login wall gak di-solve dalam batas waktu (10 menit)."""
    pass


# --- Path (SAMA dengan script metadata_video.csv yang lama) ---
ROOT_DIR = r"C:\Users\nesha\Tugas_akhir\seleksi-influencer"
DATA_DIR = os.path.join(ROOT_DIR, "01_pengumpulan_data")

PATH_COMMENTS = os.path.join(DATA_DIR, "05_scrap_komentar", "hasil_07_comments_01", "comments.csv")
PATH_METADATA_VIDEO = os.path.join(DATA_DIR, "04_1_mendapatkan_metadata_akun", "metadata_video.csv")

PROFILE_DIR = os.path.join(DATA_DIR, "00_akun_playwright", "tiktok_browser_profile")  # dipakai Cell E-TEST aja
WARNING_MP3 = os.path.join(DATA_DIR, "03_videoid_perakun", "warning.mp3")

# --- 2 AKUN TIKTOK buat scraping PARALEL (sama kayak pola 07_SCRAPE_COMMENTS_2AKUN_FINAL.py) ---
FINAL_COOKIE_DIR = os.path.join(DATA_DIR, "00_akun_playwright", "final_cookie")
PROFILE_DIR_1 = os.path.join(DATA_DIR, "00_akun_playwright", "tiktok_browser_profile_akun1")
PROFILE_DIR_2 = os.path.join(DATA_DIR, "00_akun_playwright", "tiktok_browser_profile_akun2")
COOKIES_FILE_1 = os.path.join(FINAL_COOKIE_DIR, "tiktok_cookies_akun1.json")
COOKIES_FILE_2 = os.path.join(FINAL_COOKIE_DIR, "tiktok_cookies_akun2.json")

OUTPUT_DIR = os.path.join(DATA_DIR, "04_1_mendapatkan_metadata_akun")
OUTPUT_VIEWS = os.path.join(OUTPUT_DIR, "video_views.csv")   # file BARU, ga nimpa yg lama
OUTPUT_TIDAK_ADA = os.path.join(OUTPUT_DIR, "video_tidak_ada.csv")  # video yg udah dicek & konfirmasi gak ada

VIEWS_FIELDS = ["username", "video_id", "video_url", "views", "scraped_at"]
TIDAK_ADA_FIELDS = ["username", "video_id", "video_url", "alasan", "checked_at"]

EXCLUDE_USERNAMES = ['drrichardlee']   # konsisten sama tahap-tahap sebelumnya

# --- Delay (lebih santai krn cuma scroll grid, bukan buka video 1-1) ---
DELAY_BETWEEN_SCROLL = (1.5, 3.0)
DELAY_WHEN_NO_NEW = (3.0, 6.0)   # tunggu lebih lama kalau ga ada card baru -- kasih waktu lazy-load
DELAY_BETWEEN_INFLUENCER = (20, 60)
NO_NEW_CARD_LIMIT = 20     # berhenti scroll kalau N kali berturut2 ga ada card baru (dinaikin, TikTok lazy-load kadang lambat)
MAX_SCROLL_PER_USER = 400  # safety limit, dinaikin biar cukup buat history video panjang

REALISTIC_UA = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36"
)

# --- Lock buat nulis CSV bareng-bareng dari 2 worker (sama kayak script komentar 2 akun) ---
FILE_LOCK = asyncio.Lock()

print('Config siap.')


pygame 2.6.1 (SDL 2.28.4, Python 3.13.7)
Hello from the pygame community. https://www.pygame.org/contribute.html
Config siap.


In [2]:
# ==== Cell A2: Telegram & Audio Alert ====
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

try:
    from credentials import TELEGRAM_BOT_TOKEN, TELEGRAM_CHAT_ID
    TELEGRAM_ENABLED = bool(TELEGRAM_BOT_TOKEN and TELEGRAM_CHAT_ID)
    if TELEGRAM_ENABLED:
        print("credentials.py ketemu -> Telegram alert ENABLED.")
except ImportError:
    TELEGRAM_BOT_TOKEN = None
    TELEGRAM_CHAT_ID = None
    TELEGRAM_ENABLED = False
    print("credentials.py gak ketemu -> Telegram alert DISABLED.")


def send_telegram_alert(message: str):
    if not TELEGRAM_ENABLED:
        print(f"   [Telegram disabled] pesan gak dikirim: {message[:60]}...")
        return
    try:
        url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
        payload = {"chat_id": TELEGRAM_CHAT_ID, "text": message}
        resp = requests.post(url, json=payload, timeout=10)
        data = resp.json()
        if not data.get("ok"):
            print(f"   Telegram API nolak pesan: {data}")
        else:
            print(f"   Telegram terkirim: {message[:60]}...")
    except Exception as e:
        print(f"   Gagal kirim Telegram alert: {e}")


def play_warning():
    if not os.path.exists(WARNING_MP3):
        print(f"   File {WARNING_MP3} gak ketemu")
        return
    try:
        pygame.mixer.init()
        pygame.mixer.music.load(WARNING_MP3)
        pygame.mixer.music.play()
        print("   warning.mp3 sedang diputar!")
    except Exception as e:
        print(f"   Gagal jalanin audio: {e}")


def stop_warning():
    try:
        if pygame.mixer.get_init():
            pygame.mixer.music.stop()
            pygame.mixer.quit()
    except Exception:
        pass


print('Fungsi alert siap.')


credentials.py ketemu -> Telegram alert ENABLED.
Fungsi alert siap.


---
## BLOK B — Tentuin Target: Grup `video_id` per `username`

**Cell B1**: ambil `video_id` unik dari `comments.csv`.
**Cell B2**: join `username` dari `metadata_video.csv`, grup jadi `{username: set(video_id)}`.
**Cell B3**: cek `video_id` target yang gak ketemu username-nya.
**Cell B4**: buang video_id yang udah dikonfirmasi gak ada (dari `video_tidak_ada.csv`, hasil run sebelumnya) -- biar ga dicari-cari lagi sia-sia tiap scroll.
**Cell B5**: buang video_id yang udah pernah ke-scrape (dari `video_views.csv`) -- kalau 1 username semua videonya udah selesai, usernamenya langsung dicoret, gak usah dibuka profilnya lagi di Blok F.

In [3]:
# ==== Cell B1: Video_id unik dari comments.csv ====
df_comments = pd.read_csv(PATH_COMMENTS, dtype=str, keep_default_na=False, encoding='utf-8')
if 'commenter_username' in df_comments.columns:
    df_comments = df_comments[~df_comments['commenter_username'].isin(EXCLUDE_USERNAMES)]

video_id_target = sorted(set(v for v in df_comments['video_id'].dropna().unique() if v.strip()))
print(f'Jumlah video_id unik di comments.csv: {len(video_id_target)}')


Jumlah video_id unik di comments.csv: 656


In [4]:
# ==== Cell B2: Join username, grup per username ====
df_video_meta = pd.read_csv(PATH_METADATA_VIDEO, dtype=str, keep_default_na=False, encoding='utf-8')
df_video_meta = df_video_meta[~df_video_meta['username'].isin(EXCLUDE_USERNAMES)]
df_video_meta = df_video_meta.drop_duplicates(subset='video_id', keep='last')

lookup_username = dict(zip(df_video_meta['video_id'], df_video_meta['username']))

target_per_user = {}
for vid in video_id_target:
    uname = lookup_username.get(vid)
    if uname:
        target_per_user.setdefault(uname, set()).add(vid)

print(f'Jumlah username yang punya video target: {len(target_per_user)}')
print(f'Total video target (yang ketemu username-nya): {sum(len(v) for v in target_per_user.values())}')


Jumlah username yang punya video target: 42
Total video target (yang ketemu username-nya): 651


In [5]:
# ==== Cell B3: Cek video_id target yang GAK ketemu username-nya ====
video_id_tanpa_username = [vid for vid in video_id_target if vid not in lookup_username]
print(f'Video_id target TANPA username: {len(video_id_tanpa_username)}')
if video_id_tanpa_username:
    print('Contoh (max 10):', video_id_tanpa_username[:10])
    print('>> Video ini gak bisa discrape (gak tau usernamenya) -- kasih tau Claude kalau jumlahnya banyak.')


Video_id target TANPA username: 5
Contoh (max 10): ['7593611689799503125', '7593705548093132053', '7612214747739475220', '7614068095233150229', '7638950123799612693']
>> Video ini gak bisa discrape (gak tau usernamenya) -- kasih tau Claude kalau jumlahnya banyak.


In [6]:
# ==== Cell B4: Buang video_id yang udah dikonfirmasi gak ada (dari run sebelumnya) ====
def load_tidak_ada():
    if not os.path.exists(OUTPUT_TIDAK_ADA):
        return set()
    done = set()
    with open(OUTPUT_TIDAK_ADA, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            done.add(row["video_id"])
    return done


def append_tidak_ada_rows(rows):
    if not rows:
        return
    file_exists = os.path.exists(OUTPUT_TIDAK_ADA)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(OUTPUT_TIDAK_ADA, "a", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=TIDAK_ADA_FIELDS)
        if not file_exists:
            writer.writeheader()
        writer.writerows(rows)
        f.flush()
        os.fsync(f.fileno())


video_id_tidak_ada = load_tidak_ada()
print(f'Video_id yang udah dikonfirmasi gak ada (dari run sebelumnya): {len(video_id_tidak_ada)}')

for uname in list(target_per_user.keys()):
    target_per_user[uname] -= video_id_tidak_ada
    if not target_per_user[uname]:
        del target_per_user[uname]

print(f'Setelah dibuang: {len(target_per_user)} username, '
      f'{sum(len(v) for v in target_per_user.values())} video target tersisa')


Video_id yang udah dikonfirmasi gak ada (dari run sebelumnya): 0
Setelah dibuang: 42 username, 651 video target tersisa


In [7]:
# ==== Cell B5: Buang video_id yang UDAH PERNAH ke-scrape (video_views.csv) ====
# Dicek di sini (bukan cuma pas run di Blok F) biar keliatan dari awal mana yg beneran masih
# perlu dikerjain -- kalau 1 username semua videonya udah ke-scrape, usernamenya dibuang total
# (gak usah dibuka profilnya lagi sama sekali pas Blok F).
def _load_video_id_set(path):
    if not os.path.exists(path):
        return set()
    s = set()
    with open(path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            s.add(row["video_id"])
    return s


video_id_sudah_discrape = _load_video_id_set(OUTPUT_VIEWS)
print(f'Video_id yang udah pernah ke-scrape views-nya: {len(video_id_sudah_discrape)}')

for uname in list(target_per_user.keys()):
    target_per_user[uname] -= video_id_sudah_discrape
    if not target_per_user[uname]:
        del target_per_user[uname]

print(f'Setelah dibuang: {len(target_per_user)} username, '
      f'{sum(len(v) for v in target_per_user.values())} video target tersisa (siap discrape)')


Video_id yang udah pernah ke-scrape views-nya: 610
Setelah dibuang: 6 username, 41 video target tersisa (siap discrape)


---
## BLOK C — Resume Helper + Captcha Check + Cek Eksistensi Video

**Cell C1b** (BARU): versi locked (`async with FILE_LOCK`) dari fungsi append, dipakai pas 2
worker nulis ke CSV yang sama secara bersamaan di Blok F -- biar ga tabrakan. Juga ada
`load_cookies_from_json()` buat load cookies 2 akun TikTok.
**Cell C3** itu BARU: dipakai pas ada video target yang gak ketemu pas scroll grid, buat
mastiin video itu beneran masih ada atau udah dihapus/private -- biar yang beneran gak ada
ga dicari-cari lagi tiap run.

In [8]:
# ==== Cell C1: Resume helper (load/append video_views.csv) ====
def load_existing_views():
    if not os.path.exists(OUTPUT_VIEWS):
        return set()
    done = set()
    with open(OUTPUT_VIEWS, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            done.add(row["video_id"])
    return done


def append_views_rows(rows):
    if not rows:
        return
    file_exists = os.path.exists(OUTPUT_VIEWS)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    with open(OUTPUT_VIEWS, "a", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=VIEWS_FIELDS)
        if not file_exists:
            writer.writeheader()
        writer.writerows(rows)
        f.flush()
        os.fsync(f.fileno())


scraped_views = load_existing_views()
print(f'Video yang udah ke-scrape views-nya sebelumnya: {len(scraped_views)}')


Video yang udah ke-scrape views-nya sebelumnya: 610


In [9]:
# ==== Cell C1b: Versi LOCKED dari append (dipakai pas 2 worker nulis bareng di Blok F) ====
async def append_views_rows_locked(rows):
    if not rows:
        return
    async with FILE_LOCK:
        append_views_rows(rows)


async def append_tidak_ada_rows_locked(rows):
    if not rows:
        return
    async with FILE_LOCK:
        append_tidak_ada_rows(rows)


def load_cookies_from_json(cookies_path, label=""):
    if not os.path.exists(cookies_path):
        print(f'   [{label}] Cookies file gak ketemu: {cookies_path}')
        return None
    try:
        with open(cookies_path, 'r', encoding='utf-8') as f:
            cookies = jsonlib.load(f)
        print(f'   [{label}] {len(cookies)} cookies ke-load')
        return cookies
    except Exception as e:
        print(f'   [{label}] Gagal load cookies: {e}')
        return None


print('Fungsi locked-append & load_cookies_from_json() siap.')


Fungsi locked-append & load_cookies_from_json() siap.


In [10]:
# ==== Cell C2: Login wall / captcha check (disalin dari script lama) ====
async def wait_for_login_wall_clear(page, timeout=600, context_label=""):
    blocker_selectors = [
        'text="Log in to TikTok"',
        '[data-e2e="login-modal"]',
        'div[role="dialog"]:has-text("Log in")',
        'div[role="dialog"]:has-text("Continue with")',
        'text="Drag the slider to fit the puzzle"',
        '[id*="captcha"]',
        '[class*="captcha"]',
        'iframe[src*="captcha"]',
        'img[alt="Captcha" i]',
        'img[alt*="captcha" i]',
    ]
    ALERT_REPEAT_EVERY = 30

    start = asyncio.get_event_loop().time()
    was_found = False
    alert_sent = False
    last_alert_at = None

    while asyncio.get_event_loop().time() - start < timeout:
        found = False
        for sel in blocker_selectors:
            try:
                elem = await page.query_selector(sel)
                if elem and await elem.is_visible():
                    found = True
                    break
            except Exception:
                continue

        if not found:
            if was_found:
                print("   Captcha/login wall hilang, lanjut.")
                stop_warning()
                if alert_sent:
                    send_telegram_alert(f"Captcha solved! Lanjut scraping views!\n{context_label}".strip())
                await asyncio.sleep(random.uniform(2, 4))
            return True

        now = asyncio.get_event_loop().time()
        elapsed = now - start
        remaining = int(timeout - elapsed)

        if not was_found:
            print("\n" + "=" * 70)
            print("   CAPTCHA / LOGIN WALL TERDETEKSI")
            print(f"   Timeout dalam {remaining} detik")
            print("=" * 70)
            play_warning()

        if last_alert_at is None or (now - last_alert_at) >= ALERT_REPEAT_EVERY:
            send_telegram_alert(
                f"CAPTCHA DETECTED (scraping views)!\n{context_label}\n"
                f"Sisa waktu: {remaining}s\nBuka Chrome Remote Desktop di HP buat solve."
            )
            alert_sent = True
            last_alert_at = now

        was_found = True
        print(f"   Solve dalam {remaining}s...", end="\r")
        await asyncio.sleep(3)

    print("\nTIMEOUT 10 MENIT - CAPTCHA GAK DI-SOLVE. Program dihentikan.")
    stop_warning()
    send_telegram_alert(f"CAPTCHA TIMEOUT!\n{context_label}\nData aman di CSV, resume otomatis nanti.")
    raise CaptchaTimeoutError("Captcha/login wall gak di-solve dalam 10 menit.")


print('Fungsi captcha-check siap.')


Fungsi captcha-check siap.


In [11]:
# ==== Cell C3: Cek eksistensi 1 video (dipanggil pas video target gak ketemu di grid) ====
async def is_video_unavailable_text(page):
    """Deteksi pesan 'video gak ada/gak bisa diputar' di halaman video (disalin dari script
    scraping metadata_video.csv yang lama)."""
    selectors = [
        'text=/can\\s*only\\s*be\\s*viewed\\s*in\\s*the\\s*tiktok\\s*app/i',
        'text=/only\\s*available\\s*(on|in)\\s*the\\s*tiktok\\s*app/i',
        'text=/hanya\\s*(dapat|bisa)\\s*dilihat\\s*di\\s*aplikasi\\s*tiktok/i',
        'text=/video\\s*(is\\s*)?not\\s*available/i',
        'text=/video\\s*ini\\s*tidak\\s*tersedia/i',
        'text=/this\\s*video\\s*is\\s*unavailable/i',
        'text=/page\\s*not\\s*found/i',
        'text=/couldn.?t\\s*find\\s*this\\s*account\\s*or\\s*video/i',
    ]
    for sel in selectors:
        try:
            elem = await page.query_selector(sel)
            if elem and await elem.is_visible():
                return True
        except Exception:
            continue
    return False


async def cek_video_exists(page, username, vid):
    """Buka video langsung (1x), cuma buat mastiin eksis apa ngga -- BUKAN buat ambil views
    (views cuma bisa diambil dari grid profil, sesuai temuan kamu).
    Return: ('ada' | 'tidak_ada' | 'tidak_pasti', alasan:str)"""
    url = f'https://www.tiktok.com/@{username}/video/{vid}'
    try:
        await page.goto(url, wait_until='load', timeout=30000)
    except Exception as e:
        return 'tidak_pasti', f'gagal_load: {e}'

    await asyncio.sleep(random.uniform(1.5, 2.5))
    await wait_for_login_wall_clear(page, context_label=f'cek eksistensi @{username}/{vid[:10]}')

    if await is_video_unavailable_text(page):
        return 'tidak_ada', 'pesan_video_unavailable'

    try:
        await page.wait_for_selector('[data-e2e="like-count"]', timeout=8000)
        return 'ada', 'valid_tapi_belum_ketemu_di_grid'
    except Exception:
        return 'tidak_pasti', 'timeout_nunggu_elemen_video'


print('Fungsi cek_video_exists() siap.')


Fungsi cek_video_exists() siap.


---
## BLOK E — Extract dari Grid + Scroll Loop

**Cell E1**: `extract_grid_pairs()` -- ambil semua `<a href=".../video/{id}">` di halaman, buat
tiap link cek apa di dalemnya ada `[data-e2e="video-views"]`, kalau ada pasangin
`video_id -> views_text`.
**Cell E2**: `scroll_dan_kumpulkan()` -- scroll berulang, kumpulin pasangan baru tiap scroll,
berhenti kalau semua video target punya 1 username itu ketemu / udah mentok bawah.
**Cell E-TEST**: WAJIB dites ke 1 profil dulu (belum nyimpen ke CSV, cuma nge-print).

In [12]:
# ==== Cell E1: Extract pasangan (video_id, views) dari grid yang lagi kebuka ====
async def extract_grid_pairs(page):
    """Return (pairs, anchors) -- anchors dikembaliin juga biar bisa dipakai buat scroll ke
    card PALING BAWAH yang lagi kelihatan (lihat Cell E2)."""
    pairs = {}
    anchors = await page.query_selector_all('a[href*="/video/"]')
    for a in anchors:
        try:
            href = await a.get_attribute('href')
            if not href:
                continue
            m = re.search(r'/video/(\d+)', href)
            if not m:
                continue
            vid = m.group(1)
            views_elem = await a.query_selector('[data-e2e="video-views"]')
            if views_elem:
                views_text = (await views_elem.text_content() or '').strip()
                if views_text:
                    pairs[vid] = views_text
        except Exception:
            continue
    return pairs, anchors


print('Fungsi extract_grid_pairs() siap.')


Fungsi extract_grid_pairs() siap.


In [13]:
# ==== Cell E2: Scroll berulang sambil kumpulin pasangan video_id->views ====
async def scroll_dan_kumpulkan(page, target_video_ids,
                                max_scroll=MAX_SCROLL_PER_USER, no_new_limit=NO_NEW_CARD_LIMIT):
    kumpulan = {}
    no_new_count = 0

    for i in range(max_scroll):
        pairs, anchors = await extract_grid_pairs(page)
        ada_baru = False
        for vid, views in pairs.items():
            if vid not in kumpulan:
                kumpulan[vid] = views
                ada_baru = True

        target_ketemu = target_video_ids & set(kumpulan.keys())

        if (i + 1) % 10 == 0:
            print(f'      [scroll #{i + 1}] total card ketemu: {len(kumpulan)}, '
                  f'target ketemu: {len(target_ketemu)}/{len(target_video_ids)}, '
                  f'streak-no-new: {no_new_count}/{no_new_limit}')

        if target_video_ids <= set(kumpulan.keys()):
            print(f'      Semua {len(target_video_ids)} video target ketemu di scroll #{i + 1}, stop.')
            break

        if not ada_baru:
            no_new_count += 1
            if no_new_count >= no_new_limit:
                print(f'      Mentok bawah ({no_new_limit}x berturut-turut ga ada card baru, '
                      f'scroll #{i + 1}). Target ketemu: {len(target_ketemu)}/{len(target_video_ids)}.')
                break
            # ada kemungkinan cuma lagi nunggu lazy-load, kasih jeda lebih panjang sebelum coba lagi
            await asyncio.sleep(random.uniform(*DELAY_WHEN_NO_NEW))
        else:
            no_new_count = 0

        # --- SCROLL -- pakai 3 cara sekaligus biar gak gampang 'stuck' di atas ---
        # 1) scroll_into_view_if_needed() ke card PALING BAWAH yg lagi kelihatan: Playwright
        #    otomatis nemu container yg beneran scrollable (bisa jadi bukan window/body-nya
        #    langsung, tapi div internal), jadi ini cara paling reliable.
        if anchors:
            try:
                await anchors[-1].scroll_into_view_if_needed(timeout=5000)
            except Exception:
                pass
        # 2) window.scrollBy() -- jaga2 kalau yg scroll itu emang window/body-nya langsung.
        try:
            await page.evaluate('window.scrollBy(0, 2200)')
        except Exception:
            pass
        # 3) mouse.wheel() -- fallback terakhir, posisi mouse ditaruh di tengah viewport dulu
        #    biar wheel-nya kena area konten, bukan sidebar/elemen lain.
        try:
            await page.mouse.move(700, 450)
            await page.mouse.wheel(0, 2200)
        except Exception:
            pass

        await asyncio.sleep(random.uniform(*DELAY_BETWEEN_SCROLL))

    else:
        print(f'      Kena batas MAX_SCROLL_PER_USER={max_scroll} sebelum target lengkap/mentok. '
              f'Target ketemu: {len(target_video_ids & set(kumpulan.keys()))}/{len(target_video_ids)}.')

    return kumpulan


print('Fungsi scroll_dan_kumpulkan() siap.')


Fungsi scroll_dan_kumpulkan() siap.


### Cell E-TEST — WAJIB dites ke 1 profil dulu sebelum lanjut Blok F

Ganti `TEST_USERNAME` ke username yang kamu tau punya video target. Cell ini buka profilnya,
scroll secukupnya, terus **cuma nge-print** pasangan video_id->views yang ketemu buat target
username itu (BELUM nyimpen ke CSV). Cocokin manual ke layar -- kalau cocok, lanjut ke Blok F.
Kalau kosong/aneh, kasih tau Claude.

In [14]:
# # ==== Cell E-TEST: Tes ke 1 profil ====
# TEST_USERNAME = list(target_per_user.keys())[0]   # GANTI kalau mau tes username tertentu
# test_target_ids = target_per_user[TEST_USERNAME]
# print(f'Tes username: @{TEST_USERNAME}, jumlah video target: {len(test_target_ids)}')

# async def tes_satu_profil():
#     async with async_playwright() as p:
#         context = await p.chromium.launch_persistent_context(
#             PROFILE_DIR, headless=False,
#             viewport={'width': 1400, 'height': 900}, user_agent=REALISTIC_UA,
#             args=['--disable-blink-features=AutomationControlled'],
#         )
#         page = context.pages[0] if context.pages else await context.new_page()
#         url = f'https://www.tiktok.com/@{TEST_USERNAME}'
#         print(f'Buka: {url}')
#         await page.goto(url, wait_until='load', timeout=60000)
#         await asyncio.sleep(3)
#         await wait_for_login_wall_clear(page, context_label='TEST')

#         hasil = await scroll_dan_kumpulkan(page, test_target_ids)
#         await context.close()
#         return hasil


# hasil_test = await tes_satu_profil()
# print(f'\nTotal pasangan ketemu: {len(hasil_test)}')
# cocok = {vid: v for vid, v in hasil_test.items() if vid in test_target_ids}
# print(f'Dari video TARGET yang ketemu views-nya: {len(cocok)}/{len(test_target_ids)}')
# for vid, views in list(cocok.items())[:10]:
#     print(f'  {vid} -> {views}')

# if not cocok:
#     print('\n>>> GAGAL nemu pasangan apapun. STOP -- jangan lanjut ke Blok F. Kasih tau Claude.')
# else:
#     print('\n>>> Cocokin angka di atas sama yang keliatan di layar browser. Kalau cocok, lanjut Blok F.')


---
## BLOK F — Loop Scraping PARALEL (2 Worker / 2 Akun TikTok)

Pola disalin dari `07_SCRAPE_COMMENTS_2AKUN_FINAL.py`: 2 browser context, masing-masing login
pakai cookies akun TikTok beda (`COOKIES_FILE_1` / `COOKIES_FILE_2`), kerjain SEPARUH username
masing-masing secara BERSAMAAN (`asyncio.gather`). Nulis ke CSV yang sama (`video_views.csv`,
`video_tidak_ada.csv`) pakai `FILE_LOCK` biar ga tabrakan pas 2 worker nulis bareng.

**Cell F1**: `run_worker_views()` -- isinya logic yang sama kayak sebelumnya (scroll grid,
verifikasi video gak ketemu), tapi per-worker: pakai cookies sendiri, context sendiri, nulis
pakai versi locked.
**Cell F2**: MAIN -- split `target_per_user` jadi 2 bagian (di-shuffle dulu biar beban rata),
jalanin 2 worker bareng pakai `asyncio.gather`.

Dijalanin **HANYA setelah Cell E-TEST berhasil & angkanya cocok manual**.

In [15]:
# ==== Cell F1: Worker (dipanggil 2x paralel, tiap worker pegang 1 akun TikTok) ====
async def run_worker_views(p, worker_label, profile_dir, cookies_path, usernames_subset,
                            target_per_user_dict, scraped_views_set):
    cookies = load_cookies_from_json(cookies_path, label=worker_label)
    if not cookies:
        print(f'[{worker_label}] Gak ada cookies, worker ini SKIP.')
        return 0

    os.makedirs(profile_dir, exist_ok=True)
    context = await p.chromium.launch_persistent_context(
        profile_dir, headless=False,
        viewport={'width': 1400, 'height': 900}, user_agent=REALISTIC_UA,
        args=['--disable-blink-features=AutomationControlled'],
    )
    await context.add_init_script(
        "Object.defineProperty(navigator, 'webdriver', {get: () => undefined});"
    )
    await context.add_cookies(cookies)
    print(f'[{worker_label}] {len(cookies)} cookies applied')

    page = context.pages[0] if context.pages else await context.new_page()
    try:
        await page.goto('https://www.tiktok.com', wait_until='load', timeout=30000)
        await asyncio.sleep(2)
    except Exception:
        pass

    total_baru = 0

    try:
        for idx, username in enumerate(usernames_subset, start=1):
            target_ids = target_per_user_dict[username]
            sisa_target = target_ids - scraped_views_set
            if not sisa_target:
                print(f'[{worker_label}] [{idx}/{len(usernames_subset)}] @{username}: udah ke-scrape semua, skip.')
                continue

            print(f'\n[{worker_label}] [{idx}/{len(usernames_subset)}] @{username}: '
                  f'{len(sisa_target)} video target tersisa')
            url = f'https://www.tiktok.com/@{username}'
            try:
                await page.goto(url, wait_until='load', timeout=60000)
            except Exception as e:
                print(f'   [{worker_label}] Gagal buka profil: {e}')
                continue

            await asyncio.sleep(random.uniform(2, 4))
            await wait_for_login_wall_clear(page, context_label=f'[{worker_label}] @{username}')

            hasil = await scroll_dan_kumpulkan(page, sisa_target)

            rows = []
            for vid, views in hasil.items():
                if vid in sisa_target:
                    rows.append({
                        'username': username,
                        'video_id': vid,
                        'video_url': f'https://www.tiktok.com/@{username}/video/{vid}',
                        'views': views,
                        'scraped_at': datetime.now().isoformat(),
                    })

            if rows:
                await append_views_rows_locked(rows)
                for r in rows:
                    scraped_views_set.add(r['video_id'])
                total_baru += len(rows)
                print(f'   [{worker_label}] Tersimpan {len(rows)} video baru.')

            tidak_ketemu = sisa_target - set(hasil.keys())
            if tidak_ketemu:
                MAX_CEK_PER_RUN = 5
                vids_to_check = list(tidak_ketemu)[:MAX_CEK_PER_RUN]
                sisa_belum_dicek = len(tidak_ketemu) - len(vids_to_check)
                print(f'   [{worker_label}] {len(tidak_ketemu)} video target TIDAK ketemu di grid -- '
                      f'cek eksistensinya satu-satu (maks {MAX_CEK_PER_RUN}/run)...')
                rows_tidak_ada = []
                gagal_beruntun = 0
                for vid in vids_to_check:
                    status, alasan = await cek_video_exists(page, username, vid)
                    if status == 'tidak_ada':
                        rows_tidak_ada.append({
                            'username': username, 'video_id': vid,
                            'video_url': f'https://www.tiktok.com/@{username}/video/{vid}',
                            'alasan': alasan, 'checked_at': datetime.now().isoformat(),
                        })
                        print(f'      [{worker_label}] {vid[:10]}: TIDAK ADA ({alasan}) -- gak dicari lagi run berikutnya.')
                        gagal_beruntun = 0
                    elif status == 'tidak_pasti':
                        gagal_beruntun += 1
                        print(f'      [{worker_label}] {vid[:10]}: {status} ({alasan}) -- masih jadi target run berikutnya.')
                    else:
                        gagal_beruntun = 0
                        print(f'      [{worker_label}] {vid[:10]}: {status} ({alasan}) -- masih jadi target run berikutnya.')

                    if gagal_beruntun >= 2:
                        print(f'      [{worker_label}] 2x gagal_load beruntun -- stop cek eksistensi dulu, '
                              f'kemungkinan kecepetan/kedetek. Sisa dilanjut run berikutnya.')
                        break

                    await asyncio.sleep(random.uniform(10, 20))

                if sisa_belum_dicek > 0:
                    print(f'   [{worker_label}] {sisa_belum_dicek} video lain belum dicek (kena limit {MAX_CEK_PER_RUN}/run), lanjut run berikutnya.')

                if rows_tidak_ada:
                    await append_tidak_ada_rows_locked(rows_tidak_ada)
                    print(f'   [{worker_label}] {len(rows_tidak_ada)} video dicatat sebagai TIDAK ADA.')

            if idx < len(usernames_subset):
                delay = random.uniform(*DELAY_BETWEEN_INFLUENCER)
                print(f'   [{worker_label}] Jeda {delay:.0f}s sebelum username berikutnya...')
                await asyncio.sleep(delay)

        print(f'\n[{worker_label}] SELESAI. Total video baru: {total_baru}')

    except CaptchaTimeoutError as e:
        print(f'\n[{worker_label}] PROGRAM DIHENTIKAN: {e}')
        print(f'[{worker_label}] Data yang sempat ke-scrape aman. Jalanin ulang, otomatis resume.')

    finally:
        await context.close()

    return total_baru


print('Fungsi run_worker_views() siap. Jalanin Cell F2 buat mulai scraping PARALEL beneran.')

Fungsi run_worker_views() siap. Jalanin Cell F2 buat mulai scraping PARALEL beneran.


In [18]:
# ==== Cell F2: MAIN -- split username jadi 2, jalanin 2 worker PARALEL ====
scraped_views = load_existing_views()

usernames_all = list(target_per_user.keys())
random.shuffle(usernames_all)
half = len(usernames_all) // 2
usernames_worker1 = usernames_all[:half]
usernames_worker2 = usernames_all[half:]

print(f'Total username: {len(usernames_all)}')
print(f'Worker 1: {len(usernames_worker1)} username')
print(f'Worker 2: {len(usernames_worker2)} username')

send_telegram_alert(
    f'Scraping views PARALEL mulai.\n'
    f'Worker 1: {len(usernames_worker1)} username\n'
    f'Worker 2: {len(usernames_worker2)} username'
)

async with async_playwright() as p:
    hasil_2_worker = await asyncio.gather(
        run_worker_views(p, 'WORKER-1', PROFILE_DIR_1, COOKIES_FILE_1,
                          usernames_worker1, target_per_user, scraped_views),
        run_worker_views(p, 'WORKER-2', PROFILE_DIR_2, COOKIES_FILE_2,
                          usernames_worker2, target_per_user, scraped_views),
        return_exceptions=True,
    )

total_keseluruhan = sum(r for r in hasil_2_worker if isinstance(r, int))
for r in hasil_2_worker:
    if isinstance(r, Exception):
        print(f'Salah satu worker error: {r}')

print(f'\n\nSEMUA WORKER SELESAI. Total video baru di-scrape: {total_keseluruhan}')
send_telegram_alert(f'Scraping views PARALEL selesai. Total baru: {total_keseluruhan}')

Total username: 6
Worker 1: 3 username
Worker 2: 3 username
   Telegram terkirim: Scraping views PARALEL mulai.
Worker 1: 3 username
Worker 2:...
   [WORKER-1] 18 cookies ke-load
   [WORKER-2] 21 cookies ke-load
[WORKER-2] 21 cookies applied
[WORKER-1] 18 cookies applied

[WORKER-1] [1/3] @dr.mariojohan: 2 video target tersisa

[WORKER-2] [1/3] @dr.ziee: 1 video target tersisa

   CAPTCHA / LOGIN WALL TERDETEKSI
   Timeout dalam 599 detik
   warning.mp3 sedang diputar!
   Telegram terkirim: CAPTCHA DETECTED (scraping views)!
[WORKER-2] @dr.ziee
Sisa ...
   Solve dalam 599s...
   CAPTCHA / LOGIN WALL TERDETEKSI
   Timeout dalam 599 detik
   warning.mp3 sedang diputar!
   Telegram terkirim: CAPTCHA DETECTED (scraping views)!
[WORKER-1] @dr.mariojohan...
   Captcha/login wall hilang, lanjut.
   Telegram terkirim: Captcha solved! Lanjut scraping views!
[WORKER-2] @dr.ziee...
   Captcha/login wall hilang, lanjut.
   Telegram terkirim: Captcha solved! Lanjut scraping views!
[WORKER-1] @dr.ma

In [19]:
# ==== Cell F3: Cek video yang GAGAL / masih nyangkut (buat dicek manual) ====
video_id_sudah_ada_final = _load_video_id_set(OUTPUT_VIEWS)
video_id_tidak_ada_final = load_tidak_ada()

sisa_gagal = {}
for uname, vids in target_per_user.items():
    sisa = vids - video_id_sudah_ada_final - video_id_tidak_ada_final
    if sisa:
        sisa_gagal[uname] = sisa

total_sisa = sum(len(v) for v in sisa_gagal.values())
print(f'Username yang masih punya video GAGAL/belum ke-scrape: {len(sisa_gagal)}')
print(f'Total video yang masih gagal: {total_sisa}\n')

rows_gagal = []
for uname, vids in sisa_gagal.items():
    for vid in vids:
        url = f'https://www.tiktok.com/@{uname}/video/{vid}'
        rows_gagal.append({'username': uname, 'video_id': vid, 'video_url': url})
        print(f'  @{uname}  ->  {url}')

if rows_gagal:
    df_gagal = pd.DataFrame(rows_gagal)
    path_gagal = os.path.join(OUTPUT_DIR, 'video_gagal_cek_manual.csv')
    df_gagal.to_csv(path_gagal, index=False)
    print(f'\nDisimpen ke: {path_gagal}  (buat kamu cek manual satu-satu)')
else:
    print('\nGa ada yang gagal -- semua video target udah ke-scrape atau dikonfirmasi tidak_ada.')

Username yang masih punya video GAGAL/belum ke-scrape: 4
Total video yang masih gagal: 23

  @nadialaydrus  ->  https://www.tiktok.com/@nadialaydrus/video/7602623822365265159
  @nadialaydrus  ->  https://www.tiktok.com/@nadialaydrus/video/7593325122660994311
  @nadialaydrus  ->  https://www.tiktok.com/@nadialaydrus/video/7606329965923011848
  @nadialaydrus  ->  https://www.tiktok.com/@nadialaydrus/video/7613367328067161365
  @nadialaydrus  ->  https://www.tiktok.com/@nadialaydrus/video/7608940966447205652
  @nadialaydrus  ->  https://www.tiktok.com/@nadialaydrus/video/7615956132896509204
  @nadialaydrus  ->  https://www.tiktok.com/@nadialaydrus/video/7608544288795872532
  @nadialaydrus  ->  https://www.tiktok.com/@nadialaydrus/video/7605603766322121992
  @nadialaydrus  ->  https://www.tiktok.com/@nadialaydrus/video/7607820889828101396
  @nadialaydrus  ->  https://www.tiktok.com/@nadialaydrus/video/7616749582483000597
  @nadialaydrus  ->  https://www.tiktok.com/@nadialaydrus/video/76085